# Kaikōura: post-event U-Net versus Random Forest
**Standalone, new Colab notebook.** Both models receive the same six post-event Sentinel-2 bands and two terrain layers. Target: union of the GNS V3 source and debris inventory, **not confirmed road closure**. No pre-event imagery is used.

Run cells in order. A GPU is recommended; the compact model can also run on CPU. Default 80/20/20 train/validation/test tiles is a **pilot subset**, not a full-AOI benchmark. Increase budgets only before evaluating the held-out test region. Results measure agreement with the mapped inventory; unmapped background is not proven stable ground.


In [ ]:
%pip install -q earthengine-api rasterio pyproj requests torch scikit-learn matplotlib pandas geopandas joblib


## 1. Configuration and authentication
The initialization project is for compute access, while the two asset paths identify the labels. Your authenticated account needs permission for both. Existing Earth Engine credentials are reused; if unavailable, complete the notebook authorization prompt. Never paste authentication tokens into team chat.


In [ ]:
from pathlib import Path
import json, random, math, os, time, hashlib, shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import ee, requests, rasterio, torch
from pyproj import Transformer
from rasterio.transform import Affine

SEED = 761
EE_PROJECT = 'deep-voyage-479406-j9' # Use a registered project your EE-authenticated account can access.
POST_START, POST_END = '2016-11-14', '2017-02-01' # End date exclusive; inspect image dates below.
LIMITS = {'train':80, 'val':20, 'test':20} # Pilot; increase before looking at test results.
EPOCHS, UNET_BASE, RF_MAX_PIXELS, RF_TREES = 10, 8, 150_000, 150
PATCH, SCALE, CRS = 128, 10, 'EPSG:2193'
BANDS = ['B2','B3','B4','B8','B11','B12']
SOURCE_ASSET = 'projects/deep-voyage-479406-j9/assets/kaikoura_source_v3'
DEBRIS_ASSET = 'projects/deep-voyage-479406-j9/assets/kaikoura_debris_v3'
ROOT = Path('/content/kaikoura_post_benchmark'); ROOT.mkdir(exist_ok=True, parents=True)
OUT = ROOT/'outputs'; OUT.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
try:
    ee.Initialize(project=EE_PROJECT)
    print('Reused existing Earth Engine credentials.')
except Exception as e:
    print('Please authorize an account with access to the compute project and GNS assets.')
    ee.Authenticate(auth_mode='notebook', force=True)
    ee.Initialize(project=EE_PROJECT)
print('Device:', DEVICE, '| CPU threads:', torch.get_num_threads())


## 2. Shared imagery, terrain and inventory target
QA60 masks detected cloud and cirrus; residual clouds/shadows remain possible. A median post-event composite spans weeks, so it is not immediate disaster response. SWIR bands and 30 m terrain are resampled to the 10 m output grid; this creates no additional native detail. The validity mask requires all input bands, terrain, land and AOI coverage.


In [ ]:
from datetime import datetime, timezone
roi = ee.Geometry.Rectangle([173.5,-42.5,174.0,-42.0], geodesic=False)
transformer = Transformer.from_crs(4326, CRS, always_xy=True)
corners = [transformer.transform(x,y) for x,y in [(173.5,-42.5),(174,-42.5),(173.5,-42),(174,-42)]]
xmin = math.floor(min(x for x,y in corners)/SCALE)*SCALE
ytop = math.ceil(max(y for x,y in corners)/SCALE)*SCALE
xmax, ymin = max(x for x,y in corners), min(y for x,y in corners)
GRID = [SCALE,0,xmin,0,-SCALE,ytop]
width = PATCH*SCALE
nx, ny = math.ceil((xmax-xmin)/width), math.ceil((ytop-ymin)/width)
a, b = math.floor(ny*0.60), math.floor(ny*0.80)
def split_row(row):
    if row < a-1: return 'train'
    if a <= row < b-1: return 'val'
    if row >= b: return 'test'
    return None # One complete 1.28 km tile row gap at each boundary.

def cloud_mask(image):
    qa = image.select('QA60')
    clear = qa.bitwiseAnd(1<<10).eq(0).And(qa.bitwiseAnd(1<<11).eq(0))
    return image.select(BANDS).multiply(0.0001).updateMask(clear)
collection = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filterBounds(roi)
              .filterDate(POST_START,POST_END).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',50)))
image_count = collection.size().getInfo()
if image_count == 0: raise RuntimeError('No images in the configured date range.')
post_dates = [datetime.fromtimestamp(t/1000,timezone.utc).strftime('%Y-%m-%d')
              for t in collection.aggregate_array('system:time_start').getInfo()]
optical_projection = ee.Image(collection.first()).select('B2').projection()
post = collection.map(cloud_mask).median().setDefaultProjection(optical_projection).resample('bilinear').clip(roi)
# SRTM is a pre-event terrain baseline with a native default projection.
dem_native = ee.Image('USGS/SRTMGL1_003').select('elevation')
slope_native = ee.Terrain.slope(dem_native)
terrain = dem_native.resample('bilinear').rename('elevation').addBands(slope_native.resample('bilinear').rename('slope'))
features = post.addBands(terrain).clip(roi)
source = ee.FeatureCollection(SOURCE_ASSET).filterBounds(roi)
debris = ee.FeatureCollection(DEBRIS_ASSET).filterBounds(roi)
print('Checking asset access and post-event coverage...', flush=True)
source_count, debris_count = source.size().getInfo(), debris.size().getInfo()
print('Sentinel-2 scenes:', image_count, '| dates:', sorted(set(post_dates)))
print('GNS polygons intersecting AOI:', source_count, 'source,', debris_count, 'debris')
# Assign the 10 m grid before paint; the constant-image default is otherwise one degree.
label = ee.Image.constant(0).byte().setDefaultProjection(CRS, GRID).paint(source.merge(debris),1).rename('landslide')
land = ee.Image('UMD/hansen/global_forest_change_2022_v1_10').select('datamask').eq(1)
valid = features.mask().reduce(ee.Reducer.min()).And(land).clip(roi).unmask(0,False).rename('valid')
stack = features.addBands(label).addBands(valid).unmask(0,False).toFloat()
config = {'seed':SEED,'post_start':POST_START,'post_end_exclusive':POST_END,'post_dates':post_dates,
          'bands':BANDS+['elevation','slope'],'crs':CRS,'scale':SCALE,'patch':PATCH,'grid':GRID,
          'limits':LIMITS,'epochs':EPOCHS,'unet_base':UNET_BASE,'rf_max_pixels':RF_MAX_PIXELS,'rf_trees':RF_TREES,
          'source_asset':SOURCE_ASSET,'debris_asset':DEBRIS_ASSET,
          'target':'GNS V3 source/debris union; unmapped background is not verified absence',
          'split':'north train / middle val / south test; one tile-row gaps; no overlap'}
cache_key = hashlib.sha256(json.dumps({k:config[k] for k in ['post_start','post_end_exclusive','bands','crs','scale','patch','grid','source_asset','debris_asset']},sort_keys=True).encode()).hexdigest()[:12]
DATA = ROOT/('tiles_'+cache_key); DATA.mkdir(exist_ok=True)
(ROOT/'config.json').write_text(json.dumps(config,indent=2))
print('Grid:', nx, 'columns ×', ny, 'rows; cache:', DATA, flush=True)



## 3. Spatial holdouts and cached aligned tiles
Rows are assigned to splits before looking at labels. Candidate tiles are shuffled reproducibly. A coarse validity check removes mostly ocean/cloud tiles using only input availability, never landslide density. Final downloads require ≥50% valid pixels. No positive-label oversampling of validation/test tiles. Both classes must be present in every split, or the run stops.


In [ ]:
# Coarse validity selection saves downloads of ocean/empty tiles.
coverage_file = DATA/'coverage.json'
if coverage_file.exists():
    coverage = json.loads(coverage_file.read_text())
else:
    polygons = []
    for r in range(ny):
        if split_row(r) is None: continue
        for c in range(nx):
            x, y = xmin+c*width, ytop-r*width
            geom = ee.Geometry.Rectangle([x,y-width,x+width,y], proj=CRS, geodesic=False)
            polygons.append(ee.Feature(geom,{'row':r,'col':c}))
    print('Checking input coverage for', len(polygons), 'candidate tiles...', flush=True)
    reduced = valid.reduceRegions(ee.FeatureCollection(polygons), ee.Reducer.mean(), scale=80, crs=CRS, tileScale=4)
    # Drop geometries from the returned metadata to keep this response small.
    coverage = [f['properties'] for f in reduced.map(lambda f: ee.Feature(None,f.toDictionary())).getInfo()['features']]
    coverage_file.write_text(json.dumps(coverage))
from concurrent.futures import ThreadPoolExecutor

def download_tile(candidate):
    r,c = candidate
    tile_id = f'r{r:03d}_c{c:03d}'
    path = DATA/(tile_id+'.tif')
    transform = [SCALE,0,xmin+c*width,0,-SCALE,ytop-r*width]
    if not path.exists():
        for attempt in range(3):
            try:
                url = stack.getDownloadURL({'crs':CRS,'crs_transform':transform,'dimensions':[PATCH,PATCH],
                                            'format':'GEO_TIFF','filePerBand':False})
                response = requests.get(url,timeout=180); response.raise_for_status()
                temp = path.with_suffix('.tmp'); temp.write_bytes(response.content)
                with rasterio.open(temp) as ds:
                    if ds.count != 10: raise ValueError('Expected ten stack bands.')
                temp.replace(path); break
            except Exception:
                if attempt == 2: raise
                time.sleep(2*(attempt+1))
    with rasterio.open(path) as ds:
        arr = ds.read()
        if arr.shape != (10,PATCH,PATCH): raise ValueError(f'Unexpected tile shape {arr.shape}')
        if ds.crs.to_epsg()!=2193 or not ds.transform.almost_equals(Affine(*transform)):
            raise ValueError('Tile CRS/grid mismatch.')
    mask = arr[9]>0.5
    if mask.mean()<0.50: return None
    return {'path':str(path),'tile':tile_id,'row':r,'col':c,
            'valid_pixels':int(mask.sum()),'positive_pixels':int(((arr[8]>0.5)&mask).sum())}

records = []
for split, limit in LIMITS.items():
    candidates = [(int(f['row']),int(f['col'])) for f in coverage
                  if split_row(int(f['row'])) == split and (f.get('mean') or 0) >= 0.50]
    random.Random(SEED).shuffle(candidates)
    kept = 0
    print(split, ':',len(candidates),'valid candidates; target',limit, flush=True)
    with ThreadPoolExecutor(max_workers=4) as executor:
        for start in range(0,len(candidates),4):
            if kept>=limit: break
            batch=candidates[start:start+min(4,limit-kept)]
            print('Downloading',split,'batch',start//4+1,flush=True)
            # Bounded four-request batches; consume results in deterministic candidate order.
            for record in executor.map(download_tile,batch):
                if record is None: continue
                record['split']=split;records.append(record);kept+=1
                print(f"  Ready {split} {kept}/{limit}: {record['tile']}",flush=True)
by_split = {s:[r for r in records if r['split']==s] for s in LIMITS}
summary = []
for split,items in by_split.items():
    n = sum(r['valid_pixels'] for r in items); p = sum(r['positive_pixels'] for r in items)
    summary.append({'split':split,'tiles':len(items),'valid_pixels':n,'mapped_landslide_pixels':p,'positive_fraction':p/max(n,1)})
    if len(items)<2 or p==0 or p==n:
        raise RuntimeError(f'{split} has insufficient tiles/classes; increase tile budget before evaluating, or inspect input coverage.')
    if len(items)<LIMITS[split]: print('Warning: fewer valid tiles available than requested for',split)
assert len({r['tile'] for r in records})==len(records), 'Duplicate tile across splits.'
(ROOT/'manifest.json').write_text(json.dumps(records,indent=2))
split_table = pd.DataFrame(summary); display(split_table)
fig,ax = plt.subplots(figsize=(8,5))
for split,color in [('train','#2878b5'),('val','#f49b21'),('test','#d64b4b')]:
    chosen=by_split[split];ax.scatter([r['col'] for r in chosen],[r['row'] for r in chosen],label=split,c=color,s=18)
ax.invert_yaxis();ax.set_aspect('equal');ax.set_xlabel('Grid column (west → east)');ax.set_ylabel('Grid row (north → south)');ax.legend();ax.set_title('Downloaded tiles: spatial split and gaps')
fig.savefig(OUT/'spatial_split.png',dpi=160,bbox_inches='tight');plt.show()


## 4. Models and shared preprocessing
Both use the same eight channels and fixed scaling: reflectance clipped to [0,1], elevation/3000, slope/90. GNS labels, distance to landslides and road intersection are **not inputs**. RF samples at most 150,000 training pixels; U-Net sees all valid training-tile pixels with flips/rotations. Training budgets differ and are reported. The validation split chooses checkpoints and thresholds; the test split is not used to tune models.


In [ ]:
"""Shared post-event inputs, compact U-Net, and RF benchmark helpers.

The Colab notebook embeds this module so it can run independently.
Targets are mapped GNS inventory, not confirmed road closure or stable ground.
"""
from pathlib import Path
import json
import random
import time

import numpy as np
import rasterio
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

FEATURE_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "elevation", "slope"]


def normalize_features(arr):
    """Same fixed, label-independent transform for both models."""
    x = np.nan_to_num(arr[:8].astype("float32"), nan=0, posinf=0, neginf=0)
    x[:6] = np.clip(x[:6], 0, 1)
    x[6] = np.clip(x[6], 0, 3000) / 3000.0
    x[7] = np.clip(x[7], 0, 90) / 90.0
    return x


def read_tile(record):
    with rasterio.open(record["path"]) as ds:
        arr = ds.read()
    return normalize_features(arr), (arr[8] > 0.5), (arr[9] > 0.5)


class TileDataset(Dataset):
    def __init__(self, records, augment=False):
        self.records, self.augment = records, augment

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        x, y, valid = read_tile(self.records[i])
        y, valid = y[None].astype("float32"), valid[None].astype("float32")
        if self.augment:
            if random.random() < 0.5:
                x, y, valid = x[:, :, ::-1], y[:, :, ::-1], valid[:, :, ::-1]
            if random.random() < 0.5:
                x, y, valid = x[:, ::-1, :], y[:, ::-1, :], valid[:, ::-1, :]
            k = random.randrange(4)
            x, y, valid = [np.rot90(a, k, axes=(1, 2)) for a in (x, y, valid)]
        return tuple(torch.from_numpy(a.copy()) for a in (x, y, valid))


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.GroupNorm(4, out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.GroupNorm(4, out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class PostEventUNet(nn.Module):
    def __init__(self, in_channels=8, base=8):
        super().__init__()
        self.enc = nn.ModuleList([
            ConvBlock(in_channels, base), ConvBlock(base, base * 2),
            ConvBlock(base * 2, base * 4), ConvBlock(base * 4, base * 8),
        ])
        self.pool = nn.MaxPool2d(2)
        self.dec = nn.ModuleList([
            ConvBlock(base * 8 + base * 4, base * 4),
            ConvBlock(base * 4 + base * 2, base * 2),
            ConvBlock(base * 2 + base, base),
        ])
        self.head = nn.Conv2d(base, 1, 1)

    def forward(self, x):
        skips = []
        for i, block in enumerate(self.enc):
            x = block(self.pool(x) if i else x)
            skips.append(x)
        for block, skip in zip(self.dec, reversed(skips[:-1])):
            x = nn.functional.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = block(torch.cat([x, skip], dim=1))
        return self.head(x)


def masked_loss(logits, target, valid, pos_weight):
    bce = nn.functional.binary_cross_entropy_with_logits(
        logits, target, reduction="none", pos_weight=pos_weight)
    bce = (bce * valid).sum() / valid.sum().clamp_min(1)
    p, t = torch.sigmoid(logits) * valid, target * valid
    dice_loss = 1 - (2 * (p * t).sum() + 1) / (p.sum() + t.sum() + 1)
    return bce + dice_loss


def confusion_counts(truth, probability, threshold):
    truth, pred = np.asarray(truth).astype(bool), np.asarray(probability) >= threshold
    return np.array([np.sum(~truth & ~pred), np.sum(~truth & pred),
                     np.sum(truth & ~pred), np.sum(truth & pred)], dtype="int64")


def metrics_from_counts(counts):
    tn, fp, fn, tp = [int(v) for v in counts]
    precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp,
            "precision": precision, "recall": recall,
            "f1": 2 * tp / max(2 * tp + fp + fn, 1),
            "iou": tp / max(tp + fp + fn, 1),
            "balanced_accuracy": (recall + specificity) / 2,
            "accuracy": (tp + tn) / max(sum(counts), 1),
            "positive_fraction": (tp + fn) / max(sum(counts), 1)}


def pixel_metrics(truth, probability, threshold):
    result = metrics_from_counts(confusion_counts(truth, probability, threshold))
    result["threshold"] = float(threshold)
    if len(np.unique(truth)) == 2:
        result["average_precision"] = float(average_precision_score(truth, probability))
        result["roc_auc"] = float(roc_auc_score(truth, probability))
    return result


def select_threshold(truth, probability):
    """Call on validation only. Deterministic F1 selection; test stays untouched."""
    scores = [(float(t), metrics_from_counts(confusion_counts(truth, probability, t))["f1"])
              for t in np.linspace(0.05, 0.95, 19)]
    chosen = max(scores, key=lambda pair: (pair[1], -abs(pair[0] - 0.5)))
    return chosen[0], scores


def train_unet(records_by_split, device, out, epochs=10, seed=761, base=8):
    torch.manual_seed(seed)
    random.seed(seed)
    loaders = {split: DataLoader(TileDataset(items, augment=split == "train"),
                                batch_size=8 if device.type == "cuda" else 4,
                                shuffle=split == "train", num_workers=0)
               for split, items in records_by_split.items() if split != "test"}
    model = PostEventUNet(base=base).to(device)
    npos = sum(r["positive_pixels"] for r in records_by_split["train"])
    nvalid = sum(r["valid_pixels"] for r in records_by_split["train"])
    weight = torch.tensor([min((nvalid - npos) / max(npos, 1), 20)], device=device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    best, stale, history = float("inf"), 0, []
    start = time.perf_counter()
    print(f"U-Net: {sum(p.numel() for p in model.parameters()):,} parameters; device={device}; pos_weight={float(weight.item()):.2f}", flush=True)
    for epoch in range(epochs):
        stats = {"epoch": epoch + 1}
        for split in ("train", "val"):
            model.train(split == "train")
            loss_sum, count = 0, 0
            for step, batch in enumerate(loaders[split]):
                x, y, valid = [a.to(device) for a in batch]
                with torch.set_grad_enabled(split == "train"):
                    logits = model(x)
                    loss = masked_loss(logits, y, valid, weight)
                    if split == "train":
                        optimizer.zero_grad(set_to_none=True)
                        loss.backward()
                        optimizer.step()
                loss_sum += loss.item() * len(x)
                count += len(x)
                if split == "train" and (step + 1) % 5 == 0:
                    print(f"  epoch {epoch + 1}: batch {step + 1}/{len(loaders[split])}", flush=True)
            stats[split + "_loss"] = loss_sum / max(count, 1)
        history.append(stats)
        print(f"Epoch {epoch + 1}/{epochs}: train={stats['train_loss']:.4f}, val={stats['val_loss']:.4f}", flush=True)
        if stats["val_loss"] < best - 1e-5:
            best, stale = stats["val_loss"], 0
            torch.save({"state_dict": model.state_dict(), "epoch": epoch + 1,
                        "base": base, "val_loss": best, "features": FEATURE_NAMES}, out / "unet_best.pt")
        else:
            stale += 1
        if stale >= 4:
            print("Early stopping: validation loss did not improve for four epochs.", flush=True)
            break
    checkpoint = torch.load(out / "unet_best.pt", map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    elapsed = time.perf_counter() - start
    (out / "training_history.json").write_text(json.dumps(history, indent=2))
    print(f"Best epoch {checkpoint['epoch']}; training {elapsed:.1f} seconds.", flush=True)
    return model, history, elapsed


def sample_rf_pixels(records, max_pixels=150_000, seed=761):
    rng = np.random.default_rng(seed)
    features, labels = [], []
    per_tile = max(1, int(np.ceil(max_pixels / len(records))))
    for record in records:
        x, y, valid = read_tile(record)
        indices = np.flatnonzero(valid.ravel())
        if len(indices) > per_tile:
            indices = rng.choice(indices, per_tile, replace=False)
        features.append(x.reshape(8, -1).T[indices])
        labels.append(y.ravel()[indices].astype("uint8"))
    x, y = np.concatenate(features), np.concatenate(labels)
    if len(y) > max_pixels:
        keep = rng.choice(len(y), max_pixels, replace=False)
        x, y = x[keep], y[keep]
    if len(np.unique(y)) < 2:
        raise ValueError("The sampled RF training pixels have only one class; increase RF_MAX_PIXELS.")
    return x, y


def train_rf(records, out, max_pixels=150_000, trees=150, seed=761):
    import joblib
    x, y = sample_rf_pixels(records, max_pixels, seed)
    model = RandomForestClassifier(n_estimators=trees, max_depth=20,
        min_samples_leaf=5, max_features="sqrt", class_weight="balanced_subsample",
        n_jobs=-1, random_state=seed)
    print(f"RF: {len(y):,} training pixels; mapped-positive fraction={y.mean():.4f}; trees={trees}", flush=True)
    start = time.perf_counter()
    model.fit(x, y)
    elapsed = time.perf_counter() - start
    joblib.dump(model, out / "rf_model.joblib")
    print(f"RF training {elapsed:.1f} seconds.", flush=True)
    return model, elapsed, {"sampled_pixels": int(len(y)), "positive_pixels": int(y.sum())}


def tile_probabilities(record, unet, rf, device):
    x, y, valid = read_tile(record)
    with torch.no_grad():
        p_unet = torch.sigmoid(unet(torch.from_numpy(x)[None].to(device)))[0, 0].cpu().numpy()
    p_rf = np.zeros(y.shape, dtype="float32")
    if valid.any():
        features = x.reshape(8, -1).T[valid.ravel()]
        p_rf[valid] = rf.predict_proba(features)[:, list(rf.classes_).index(1)]
    return {"truth": y, "valid": valid, "U-Net": p_unet, "Random Forest": p_rf}


def predict_split(records, unet, rf, device):
    results = []
    for i, record in enumerate(records):
        results.append(tile_probabilities(record, unet, rf, device))
        if (i + 1) % 5 == 0 or i + 1 == len(records):
            print(f"Predicted {i + 1}/{len(records)} {record['split']} tiles", flush=True)
    return results


def pooled(results, key):
    return np.concatenate([item[key][item["valid"]] for item in results])


def bootstrap_f1(results, key, threshold, seed=761, repetitions=1000):
    """Tile-resampling interval; spatial dependence still limits interpretation."""
    counts = np.stack([confusion_counts(item["truth"][item["valid"]],
                       item[key][item["valid"]], threshold) for item in results])
    rng = np.random.default_rng(seed)
    values = [metrics_from_counts(counts[rng.integers(0, len(counts), len(counts))].sum(0))["f1"]
              for _ in range(repetitions)]
    return [float(v) for v in np.percentile(values, [2.5, 97.5])]


In [ ]:
unet, history, unet_seconds = train_unet(by_split, DEVICE, OUT, epochs=EPOCHS, seed=SEED, base=UNET_BASE)
rf, rf_seconds, rf_budget = train_rf(by_split['train'], OUT, max_pixels=RF_MAX_PIXELS, trees=RF_TREES, seed=SEED)
fig,axes = plt.subplots(1,2,figsize=(11,4))
axes[0].plot([h['epoch'] for h in history],[h['train_loss'] for h in history],label='training')
axes[0].plot([h['epoch'] for h in history],[h['val_loss'] for h in history],label='validation')
axes[0].set_xlabel('Epoch');axes[0].set_ylabel('Masked BCE + Dice');axes[0].legend();axes[0].set_title('U-Net learning curve')
order=np.argsort(rf.feature_importances_)
axes[1].barh(np.array(FEATURE_NAMES)[order],rf.feature_importances_[order]);axes[1].set_title('RF impurity importance (descriptive)')
fig.tight_layout();fig.savefig(OUT/'training_and_rf_importance.png',dpi=160);plt.show()


## 5. Select thresholds on validation, then evaluate test once
Precision asks how many predicted landslide pixels agree with the inventory; recall asks how many mapped pixels were found. F1 balances them; IoU measures overlap. Balanced accuracy handles imbalance better than ordinary accuracy. Average precision assesses probability ranking. A background-only baseline makes class imbalance visible. Tile-bootstrap F1 intervals are approximate; neighboring tiles are not independent.


In [ ]:
val_predictions = predict_split(by_split['val'],unet,rf,DEVICE)
val_truth = pooled(val_predictions,'truth')
thresholds, val_curves = {}, {}
for name in ['Random Forest','U-Net']:
    thresholds[name],val_curves[name] = select_threshold(val_truth,pooled(val_predictions,name))
    print(name,'validation-selected threshold:',round(thresholds[name],2))
(OUT/'thresholds.json').write_text(json.dumps(thresholds,indent=2))
# Do not alter model, split or thresholds after viewing the following test results.
test_predictions = predict_split(by_split['test'],unet,rf,DEVICE)
test_truth = pooled(test_predictions,'truth')
metrics = {}
for name in ['Random Forest','U-Net']:
    m = pixel_metrics(test_truth,pooled(test_predictions,name),thresholds[name])
    m['f1_tile_bootstrap_95_interval'] = bootstrap_f1(test_predictions,name,thresholds[name],SEED)
    m['training_seconds'] = rf_seconds if name=='Random Forest' else unet_seconds
    metrics[name] = m
metrics['All background'] = pixel_metrics(test_truth,np.zeros(len(test_truth)),0.5)
metrics['metadata'] = {'test_tiles':len(test_predictions),'valid_test_pixels':int(len(test_truth)),
    'rf_training_budget':rf_budget,'unet_valid_training_pixels':int(split_table.loc[split_table['split']=='train','valid_pixels'].iloc[0]),
    'target':'agreement with GNS mapped landslides, not road blockage','threshold_selection':'validation F1'}
(OUT/'test_metrics.json').write_text(json.dumps(metrics,indent=2))
columns=['precision','recall','f1','iou','balanced_accuracy','accuracy','average_precision','roc_auc','threshold','training_seconds']
metric_table=pd.DataFrame({n:{k:v for k,v in m.items() if k in columns} for n,m in metrics.items() if n!='metadata'}).T
metric_table.to_csv(OUT/'test_metrics.csv'); display(metric_table.round(4))
fig,axes=plt.subplots(1,3,figsize=(15,4))
for name in ['Random Forest','U-Net']:
    curve=val_curves[name];axes[0].plot([v[0] for v in curve],[v[1] for v in curve],label=name)
axes[0].set_xlabel('Threshold');axes[0].set_ylabel('Validation F1');axes[0].legend();axes[0].set_title('Validation-only threshold selection')
for ax,name in zip(axes[1:],['Random Forest','U-Net']):
    m=metrics[name]; cm=np.array([[m['tn'],m['fp']],[m['fn'],m['tp']]])
    ax.imshow(cm,cmap='Blues')
    for (r,c),value in np.ndenumerate(cm):ax.text(c,r,f'{value:,}',ha='center',va='center',color='black')
    ax.set_xticks([0,1],['Background','Landslide']);ax.set_yticks([0,1],['Background','Landslide']);ax.set_xlabel('Predicted');ax.set_ylabel('GNS target');ax.set_title(name+' — held-out test')
fig.tight_layout();fig.savefig(OUT/'thresholds_and_confusion.png',dpi=160);plt.show()


## 6. Matched qualitative comparisons
The same held-out tiles are shown for both models: RGB, GNS target, RF probability, U-Net probability, RF errors, U-Net errors. Green = correct positive, red = false positive, blue = missed mapped positive; invalid pixels are gray. Examples follow a fixed grid order, not selected to favor a model. Inspect full test tiles in exported files; the overview can miss small failures.


In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
order=sorted(range(len(by_split['test'])),key=lambda i:(by_split['test'][i]['row'],by_split['test'][i]['col']))[:4]
fig,axes=plt.subplots(len(order),6,figsize=(19,3.4*len(order)),squeeze=False)
error_cmap=ListedColormap(['#e7e7e7','#3aa657','#db4437','#3367d6'])
error_cmap.set_bad('#888888')
prob_cmap=plt.get_cmap('viridis').copy(); prob_cmap.set_bad('#888888')
for row,index in enumerate(order):
    record,item=by_split['test'][index],test_predictions[index]
    x,_,_=read_tile(record); valid=item['valid'];truth=item['truth']
    rgb=np.clip(x[[2,1,0]].transpose(1,2,0)/0.30,0,1);rgb[~valid]=0.5
    images=[rgb,np.ma.masked_where(~valid,truth),np.ma.masked_where(~valid,item['Random Forest']),np.ma.masked_where(~valid,item['U-Net'])]
    for name in ['Random Forest','U-Net']:
        pred=item[name]>=thresholds[name]
        error=np.zeros(truth.shape,dtype='uint8');error[pred&truth]=1;error[pred&~truth]=2;error[~pred&truth]=3
        images.append(np.ma.masked_where(~valid,error))
    titles=['Post-event RGB','GNS target','RF probability','U-Net probability','RF: TP / FP / FN','U-Net: TP / FP / FN']
    for col,(image,title) in enumerate(zip(images,titles)):
        if col==0:axes[row,col].imshow(image)
        elif col>=4:axes[row,col].imshow(image,cmap=error_cmap,vmin=0,vmax=3,interpolation='nearest')
        else:axes[row,col].imshow(image,cmap=prob_cmap,vmin=0,vmax=1)
        axes[row,col].set_title(title if row==0 else '');axes[row,col].axis('off')
    axes[row,0].text(0,1.03,record['tile'],transform=axes[row,0].transAxes,fontsize=9)
fig.legend(handles=[Patch(color='#3aa657',label='Correct landslide (TP)'),Patch(color='#db4437',label='False positive (FP)'),Patch(color='#3367d6',label='Missed mapped landslide (FN)'),Patch(color='#888888',label='Invalid')],loc='lower center',ncol=4)
fig.tight_layout(rect=(0,0.035,1,1));fig.savefig(OUT/'matched_test_comparison.png',dpi=180,bbox_inches='tight');plt.show()
fig,ax=plt.subplots(figsize=(7,4))
metric_table.loc[['Random Forest','U-Net'],['precision','recall','f1','iou','balanced_accuracy']].T.plot.bar(ax=ax)
ax.set_ylim(0,1);ax.set_ylabel('Score against GNS inventory');ax.set_title('Held-out spatial test: identical valid pixels');fig.tight_layout();fig.savefig(OUT/'metric_comparison.png',dpi=160);plt.show()

# Additional label-selected examples help inspect actual segmentation boundaries.
# Selection uses GNS target density only, never model performance; aggregate metrics stay unchanged.
positive_order=sorted(range(len(by_split['test'])),key=lambda i:(-by_split['test'][i]['positive_pixels'],by_split['test'][i]['tile']))[:3]
fig,axes=plt.subplots(len(positive_order),4,figsize=(12,3.5*len(positive_order)),squeeze=False)
mask_cmap=ListedColormap(['#222222','#ffbf00']);mask_cmap.set_bad('#888888')
for row,index in enumerate(positive_order):
    record,item=by_split['test'][index],test_predictions[index]
    x,_,_=read_tile(record);v=item['valid']
    rgb=np.clip(x[[2,1,0]].transpose(1,2,0)/0.30,0,1);rgb[~v]=0.5
    arrays=[rgb,item['truth'],item['Random Forest']>=thresholds['Random Forest'],item['U-Net']>=thresholds['U-Net']]
    titles=['Post-event RGB','GNS mapped mask','RF predicted mask','U-Net predicted mask']
    for col,array in enumerate(arrays):
        if col==0:axes[row,col].imshow(array)
        else:axes[row,col].imshow(np.ma.masked_where(~v,array),cmap=mask_cmap,vmin=0,vmax=1,interpolation='nearest')
        axes[row,col].set_title(titles[col] if row==0 else '');axes[row,col].axis('off')
    axes[row,0].text(0,1.03,record['tile'],transform=axes[row,0].transAxes,fontsize=9)
fig.suptitle('Test examples with most mapped landslide pixels (label-selected, matched models)',fontsize=11)
fig.tight_layout();fig.savefig(OUT/'binary_mask_comparison.png',dpi=180,bbox_inches='tight');plt.show()


## 7. Georeferenced test predictions and feedback
Exports cover **downloaded test tiles only**, not the whole AOI. Probability and binary masks preserve NZTM grid/CRS; invalid pixels use nodata. GeoJSON landslide polygons can feed Component 2, but this subset is insufficient for complete routing. Feedback below reports observed failures without changing the held-out test protocol.


In [ ]:
from rasterio.features import shapes
from shapely.geometry import shape
import geopandas as gpd
pred_dir=OUT/'test_predictions';pred_dir.mkdir(exist_ok=True)
polygons={name:[] for name in thresholds}
for record,item in zip(by_split['test'],test_predictions):
    with rasterio.open(record['path']) as ds: profile=ds.profile.copy()
    valid=item['valid']
    for name,threshold in thresholds.items():
        stem='rf' if name=='Random Forest' else 'unet'
        probability=item[name].astype('float32'); mask=(probability>=threshold).astype('uint8')
        profile.update(count=1,dtype='float32',nodata=-9999,compress='deflate')
        with rasterio.open(pred_dir/f"{record['tile']}_{stem}_probability.tif",'w',**profile) as ds:
            ds.write(np.where(valid,probability,-9999).astype('float32'),1)
        profile.update(dtype='uint8',nodata=255)
        with rasterio.open(pred_dir/f"{record['tile']}_{stem}_mask.tif",'w',**profile) as ds:
            ds.write(np.where(valid,mask,255).astype('uint8'),1)
        for geometry,value in shapes(mask,mask=valid&(mask==1),transform=profile['transform']):
            if value==1:polygons[name].append({'tile':record['tile'],'model':name,'geometry':shape(geometry)})
for name,items in polygons.items():
    stem='rf' if name=='Random Forest' else 'unet'
    path=OUT/(stem+'_test_landslides.geojson')
    if items:gpd.GeoDataFrame(items,crs=CRS).to_crs(4326).to_file(path,driver='GeoJSON')
    else:path.write_text(json.dumps({'type':'FeatureCollection','features':[]}))
feedback=[]
for name in ['Random Forest','U-Net']:
    m=metrics[name]
    feedback.append(f"{name}: test F1={m['f1']:.3f}, IoU={m['iou']:.3f}, precision={m['precision']:.3f}, recall={m['recall']:.3f}.")
    if m['precision']<0.5:feedback.append(f'{name}: many predicted positives disagree with the inventory; inspect bare rock, shadows and inventory omissions.')
    if m['recall']<0.5:feedback.append(f'{name}: many mapped positives are missed; inspect small landslides, mixed pixels and the spatial domain shift.')
feedback += ['Pilot spatial holdout only; no claim of full-region or road-passability accuracy.',
             'GNS inventory is event-specific, while background may include omitted/older slides.',
             'Do not tune on this test region; use validation or a new independent event for further experiments.']
print('\n'.join(feedback));(OUT/'feedback.txt').write_text('\n'.join(feedback))
shutil.copy(ROOT/'config.json',OUT/'config.json');shutil.copy(ROOT/'manifest.json',OUT/'manifest.json')
split_table.to_csv(OUT/'split_summary.csv',index=False)
print('Saved figures, metrics, models and held-out georeferenced predictions:',OUT)


In [ ]:
bundle=shutil.make_archive(str(ROOT/'kaikoura_post_unet_rf_outputs'),'zip',OUT)
print('Result bundle:',bundle)
# Run this final cell when ready to download results.
from google.colab import files
files.download(bundle)


## Interpretation and sources
- Imagery: Copernicus Sentinel-2 L1C, Earth Engine `COPERNICUS/S2_HARMONIZED`; [official catalog](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED).
- Terrain: NASA SRTM, `USGS/SRTMGL1_003`; [official catalog](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003).
- Labels: GNS Kaikōura V3 source/debris inventory, DOI [10.17603/ds2-rpyg-0h84](https://doi.org/10.17603/ds2-rpyg-0h84); retain the supplied authors/citation and attribution.
- Eight common inputs and shared train/validation/test regions make comparison interpretable; RF uses a bounded pixel sample and U-Net spatial context, so this is not equal compute-budget optimization.
- Validation-selected F1 thresholds, compact U-Net and modest RF defaults are pilot choices. High ordinary accuracy can reflect background imbalance; prioritize F1/IoU/precision/recall and inspect the error maps.
- The task is mapped-landslide segmentation after the event. Neither model establishes actual road blockage, safe passage, vehicle clearance or real-time availability.


In [ ]:
# Append after the existing Kaikoura_PostEvent_UNet_RF pilot.
# Cloud training only; the original RF model and test results stay unchanged.
from IPython.display import display, Markdown
import joblib, logging, warnings
from sklearn.ensemble import RandomForestClassifier

required = ['by_split', 'sample_rf_pixels', 'read_tile', 'pixel_metrics',
            'select_threshold', 'SEED', 'RF_MAX_PIXELS', 'OUT', 'ROOT']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError('Run the pilot setup and data/helper cells first: ' + ', '.join(missing))

# Suppress only the known ten-band TIFF color-metadata warning, not read failures.
class _TiffColorMetadataFilter(logging.Filter):
    def filter(self, record):
        return 'Sum of Photometric type-related color channels and ExtraSamples' not in record.getMessage()
logging.getLogger('rasterio._env').addFilter(_TiffColorMetadataFilter())

display(Markdown('## Random Forest tree-count stability — validation only\n'
                 '100 → 150 → 300 trees, identical sampled training pixels, seed and other parameters. '
                 'Fixed-threshold metrics isolate tree-count changes. Separately retuned thresholds use '
                 'validation only. No test-region inference or model selection uses test scores.'))

SWEEP_OUT = ROOT/'rf_tree_stability'; SWEEP_OUT.mkdir(exist_ok=True)
tree_counts = [100, 150, 300]
fixed_threshold = float(globals().get('thresholds', {}).get('Random Forest', 0.30))
print('Fixed common threshold:', fixed_threshold, '(previous validation-selected pilot threshold)', flush=True)
rf_sweep_x, rf_sweep_y = sample_rf_pixels(by_split['train'], RF_MAX_PIXELS, SEED)
validation_x, validation_y, validation_tile_ids = [], [], []
for record in by_split['val']:
    x, y, valid = read_tile(record)
    validation_x.append(x.reshape(8,-1).T[valid.ravel()])
    validation_y.append(y[valid])
    validation_tile_ids.append(np.full(int(valid.sum()),record['tile']))
validation_x = np.concatenate(validation_x)
validation_y = np.concatenate(validation_y)
validation_tile_ids = np.concatenate(validation_tile_ids)
if len(np.unique(validation_y)) != 2:
    raise RuntimeError('Validation must include both mapped landslide and background pixels.')

rf_growth = RandomForestClassifier(n_estimators=100, max_depth=20,
    min_samples_leaf=5, max_features='sqrt', class_weight='balanced_subsample',
    n_jobs=-1, random_state=SEED, warm_start=True)
rf_stability_rows, rf_sweep_probabilities = [], {}
cumulative_seconds, previous_probability = 0.0, None
for trees in tree_counts:
    rf_growth.set_params(n_estimators=trees)
    started = time.perf_counter()
    with warnings.catch_warnings():
        # Exact same training data are reused, so this changing-data caution does not apply.
        warnings.filterwarnings('ignore', message='class_weight presets.*', category=UserWarning)
        rf_growth.fit(rf_sweep_x, rf_sweep_y)
    fit_seconds = time.perf_counter()-started
    cumulative_seconds += fit_seconds
    probability = rf_growth.predict_proba(validation_x)[:,list(rf_growth.classes_).index(1)]
    rf_sweep_probabilities[trees] = probability.copy()
    fixed = pixel_metrics(validation_y, probability, fixed_threshold)
    optimized_threshold, _ = select_threshold(validation_y, probability)
    optimized = pixel_metrics(validation_y, probability, optimized_threshold)
    row = {'trees': trees, 'fixed_threshold': fixed_threshold,
           'precision': fixed['precision'], 'recall': fixed['recall'],
           'f1': fixed['f1'], 'iou': fixed['iou'],
           'average_precision': fixed['average_precision'], 'roc_auc': fixed['roc_auc'],
           'validation_selected_threshold': optimized_threshold,
           'retuned_f1': optimized['f1'], 'retuned_iou': optimized['iou'],
           'incremental_fit_seconds': fit_seconds, 'cumulative_fit_seconds': cumulative_seconds,
           'probability_change_mean_abs': None if previous_probability is None else
               float(np.mean(np.abs(probability-previous_probability))),
           'fixed_mask_change_fraction': None if previous_probability is None else
               float(np.mean((probability>=fixed_threshold)!=(previous_probability>=fixed_threshold)))}
    rf_stability_rows.append(row)
    # Save before the next fit mutates the same warm-start forest.
    joblib.dump(rf_growth, SWEEP_OUT/f'rf_{trees}_trees.joblib')
    np.savez_compressed(SWEEP_OUT/f'validation_probabilities_{trees}.npz',
                        probability=probability, truth=validation_y, tile=validation_tile_ids)
    previous_probability = probability.copy()
    print(f"{trees} trees complete: validation F1={fixed['f1']:.4f}, IoU={fixed['iou']:.4f}, "
          f"AP={fixed['average_precision']:.4f}; cumulative fit={cumulative_seconds:.1f}s", flush=True)

# The 150-tree stage should reproduce the independently fitted pilot forest.
pilot_max_abs_difference = None
if 'val_predictions' in globals():
    pilot_probability = pooled(val_predictions,'Random Forest')
    pilot_max_abs_difference = float(np.max(np.abs(rf_sweep_probabilities[150]-pilot_probability)))
    if not np.allclose(rf_sweep_probabilities[150], pilot_probability, atol=1e-6, rtol=0):
        raise RuntimeError('150-tree validation probabilities differ from the pilot; check data and parameters.')
    print(f'150-tree pilot reproducibility check passed (max probability difference {pilot_max_abs_difference:.2g}).')

rf_stability_table = pd.DataFrame(rf_stability_rows).set_index('trees')
rf_stability_table.to_csv(SWEEP_OUT/'rf_tree_stability_validation.csv')
display(rf_stability_table[['precision','recall','f1','iou','average_precision',
                           'validation_selected_threshold','retuned_f1','cumulative_fit_seconds']].round(4))
fig, axes = plt.subplots(1,2,figsize=(12,4))
for metric in ['f1','iou','average_precision']:
    axes[0].plot(tree_counts,rf_stability_table[metric],marker='o',label=metric)
axes[0].set_title(f'Validation metrics, fixed threshold {fixed_threshold:.2f}')
axes[0].set_ylabel('Score'); axes[0].legend()
axes[1].plot(tree_counts,rf_stability_table['f1'],marker='o',label='F1: common threshold')
axes[1].plot(tree_counts,rf_stability_table['retuned_f1'],marker='o',label='F1: validation-retuned threshold')
axes[1].set_title('Threshold sensitivity on the same validation region')
axes[1].set_ylabel('F1');axes[1].legend()
for ax in axes:
    ax.set_xlabel('Number of trees');ax.set_xticks(tree_counts);ax.grid(alpha=.25)
fig.suptitle('RF tree-count stability: same pixels, parameters and seed; validation only')
fig.tight_layout();fig.savefig(SWEEP_OUT/'rf_tree_stability.png',dpi=160);plt.show()

change = rf_stability_table.loc[300,['f1','iou','average_precision']]-rf_stability_table.loc[150,['f1','iou','average_precision']]
rf_stability_feedback = (
    f"150 -> 300 trees, fixed threshold {fixed_threshold:.2f}: "
    f"validation delta F1={change['f1']:+.6f}, delta IoU={change['iou']:+.6f}, "
    f"delta AP={change['average_precision']:+.6f}.\n"
    'This is one fixed-seed, nested-forest experiment. Metric stability is not proof of high accuracy '
    'or robustness across seeds/regions. All model comparisons here use validation only.\n'
    'Keep the existing pilot test outputs unchanged; do not use this validation sweep as final generalization evidence.\n')
print(rf_stability_feedback)
(SWEEP_OUT/'feedback.txt').write_text(rf_stability_feedback)
metadata = {'tree_counts':tree_counts,'seed':SEED,'training_pixels':len(rf_sweep_y),
    'validation_pixels':len(validation_y),'training_data_sha256':
        hashlib.sha256(rf_sweep_x.tobytes()+rf_sweep_y.tobytes()).hexdigest(),
    'train_tiles':[r['tile'] for r in by_split['train']],
    'validation_tiles':[r['tile'] for r in by_split['val']],
    'fixed_threshold':fixed_threshold,'pilot_150_probability_max_abs_difference':pilot_max_abs_difference,
    'model_params':rf_growth.get_params(),'evaluation':'validation only; original test results preserved'}
(SWEEP_OUT/'rf_tree_stability_metadata.json').write_text(json.dumps(metadata,indent=2))
rf_stability_bundle = shutil.make_archive(str(ROOT/'kaikoura_rf_tree_stability_outputs'),'zip',SWEEP_OUT)
print('Result bundle:',rf_stability_bundle)
from google.colab import files
files.download(rf_stability_bundle)


In [ ]:
"""Validation-only, controlled U-Net continuation experiments for Colab.

Execute this source in the benchmark notebook after its helper-definition cell.
Required existing helpers: PostEventUNet, TileDataset, masked_loss.
No test records or test metrics are read. No training is run when this file loads.
The architecture, eight inputs, BCE+Dice and AdamW defaults stay unchanged.
"""
from pathlib import Path
import csv
import hashlib
import json
import random
import time

import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import average_precision_score


def sweep_threshold_metrics(truth, probability):
    """Exact counts on a shared 0.01–0.99 grid without 99 large masks."""
    truth = np.asarray(truth, dtype=bool).reshape(-1)
    probability = np.asarray(probability).reshape(-1)
    thresholds = np.linspace(0.01, 0.99, 99)
    # Equality belongs to the upper bin, matching probability >= threshold.
    bins = np.searchsorted(thresholds, probability, side="right")
    positive = np.bincount(bins[truth], minlength=100).astype("int64")
    negative = np.bincount(bins[~truth], minlength=100).astype("int64")
    tp = np.cumsum(positive[::-1])[::-1][1:]
    fp = np.cumsum(negative[::-1])[::-1][1:]
    fn = int(truth.sum()) - tp
    tn = int((~truth).sum()) - fp
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    # Same F1 tie break as the original helper: nearest threshold to 0.5.
    index = max(range(len(thresholds)), key=lambda i: (float(f1[i]), -abs(float(thresholds[i]) - 0.5)))
    chosen = {
        "val_threshold": float(thresholds[index]),
        "val_f1": float(f1[index]),
        "val_precision": float(tp[index] / max(int(tp[index] + fp[index]), 1)),
        "val_recall": float(tp[index] / max(int(tp[index] + fn[index]), 1)),
        "val_iou": float(tp[index] / max(int(tp[index] + fp[index] + fn[index]), 1)),
        "val_tn": int(tn[index]), "val_fp": int(fp[index]),
        "val_fn": int(fn[index]), "val_tp": int(tp[index]),
    }
    curve = [{"threshold": float(t), "f1": float(score), "tn": int(n),
              "fp": int(p), "fn": int(m), "tp": int(h)}
             for t, score, n, p, m, h in zip(thresholds, f1, tn, fp, fn, tp)]
    return chosen, curve


def _sweep_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2))
    temporary.replace(path)


def _sweep_torch_save(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temporary)
    temporary.replace(path)


def _sweep_rng_state(train_generator, val_generator):
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
            "train_loader": train_generator.get_state(),
            "val_loader": val_generator.get_state()}


def _sweep_restore_rng(state, train_generator, val_generator):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"].cpu())
    if torch.cuda.is_available() and state["cuda"]:
        torch.cuda.set_rng_state_all([item.cpu() for item in state["cuda"]])
    train_generator.set_state(state["train_loader"].cpu())
    val_generator.set_state(state["val_loader"].cpu())


def _sweep_rank(record):
    # AP is primary; F1 breaks exact AP ties; earlier epoch breaks exact metric ties.
    return (record["val_ap"], record["val_f1"], -record["epoch"])


def _sweep_data_fingerprint(records_by_split):
    """Only train/validation files are read; content hashes freeze the data."""
    manifest = []
    for split in ("train", "val"):
        for record in records_by_split[split]:
            path = Path(record["path"])
            digest = hashlib.sha256()
            with path.open("rb") as handle:
                for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                    digest.update(chunk)
            manifest.append({"split": split, "tile": record.get("tile", path.stem),
                "sha256": digest.hexdigest(), "valid_pixels": int(record["valid_pixels"]),
                "positive_pixels": int(record["positive_pixels"])})
    encoded = json.dumps(manifest, sort_keys=True).encode()
    return hashlib.sha256(encoded).hexdigest(), manifest


def run_validation_sweep(records_by_split, device, out_root, *,
                         pos_weights=(20.0, 5.0, 1.0), max_epochs=150,
                         patience=20, seed=761, base=8, expected_counts=(80, 20),
                         patience_ap_min_delta=1e-4, resume=True, unet_class=None, dataset_class=None,
                         loss_function=None):
    """Train controlled runs; select top-three checkpoints using validation only.

    Call after the benchmark helpers are defined, or explicitly pass the helper
    class/function arguments when importing this module. Resume checkpoints contain
    optimizer, RNG, loader-generator, ranking, history and configuration state.
    Latest checkpoints are overwritten atomically every epoch, not just every ten.
    Validation loss is logged for diagnosis; losses from different weights should
    not be used to rank the runs because their objective scaling differs.
    """
    if unet_class is None:
        unet_class = globals().get("PostEventUNet")
    if dataset_class is None:
        dataset_class = globals().get("TileDataset")
    if loss_function is None:
        loss_function = globals().get("masked_loss")
    if any(helper is None for helper in (unet_class, dataset_class, loss_function)):
        raise RuntimeError("Run the benchmark helper-definition cell first, or pass helper classes/functions explicitly.")
    if base != 8:
        raise ValueError("This controlled sweep freezes the existing base=8 architecture.")
    if not (1 <= max_epochs <= 150) or patience < 1:
        raise ValueError("Use 1–150 epochs and a positive patience.")
    if tuple(float(weight) for weight in pos_weights) != (20.0, 5.0, 1.0):
        raise ValueError("The agreed controlled sweep uses pos_weight 20, 5 and 1 in that order.")
    for split, expected in zip(("train", "val"), expected_counts):
        items = records_by_split[split]
        if len(items) != expected:
            raise ValueError(f"Frozen {split} split should contain {expected} tiles, found {len(items)}.")
        valid = sum(int(item["valid_pixels"]) for item in items)
        positive = sum(int(item["positive_pixels"]) for item in items)
        if not 0 < positive < valid:
            raise ValueError(f"{split} must contain both mapped-positive and background valid pixels.")
    device = torch.device(device)
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    fingerprint, frozen_manifest = _sweep_data_fingerprint(records_by_split)
    _sweep_json(out_root / "frozen_train_val_manifest.json", frozen_manifest)
    config_common = {"seed": seed, "base": base, "max_epochs": max_epochs,
        "patience": patience, "patience_ap_min_delta": patience_ap_min_delta, "data_fingerprint": fingerprint,
        "train_tiles": len(records_by_split["train"]), "val_tiles": len(records_by_split["val"]),
        "optimizer": "AdamW", "lr": 1e-3, "weight_decay": 1e-4,
        "loss": "masked weighted BCE + masked Dice (unchanged)",
        "ranking": "validation AP, then validation F1, then earlier epoch",
        "thresholds": [float(t) for t in np.linspace(0.01, 0.99, 99)],
        "test_usage": "none"}
    _sweep_json(out_root / "sweep_config.json", config_common)
    # Cache train/validation tensors once (~65 MB for 100 standard tiles).
    # Exactly the same Python-random flips/rotations as the existing TileDataset.
    cached = {}
    for split in ("train", "val"):
        source_dataset = dataset_class(records_by_split[split], augment=False)
        cached[split] = [tuple(t.detach().cpu().numpy().copy() for t in source_dataset[i])
                         for i in range(len(source_dataset))]
    class CachedSweepDataset(Dataset):
        def __init__(self, arrays, augment=False):
            self.arrays, self.augment = arrays, augment
        def __len__(self):
            return len(self.arrays)
        def __getitem__(self, index):
            x, y, valid = self.arrays[index]
            if self.augment:
                if random.random() < 0.5:
                    x, y, valid = x[:, :, ::-1], y[:, :, ::-1], valid[:, :, ::-1]
                if random.random() < 0.5:
                    x, y, valid = x[:, ::-1, :], y[:, ::-1, :], valid[:, ::-1, :]
                k = random.randrange(4)
                x, y, valid = [np.rot90(a, k, axes=(1, 2)) for a in (x, y, valid)]
            return tuple(torch.from_numpy(a.copy()) for a in (x, y, valid))
    all_results = []
    # cuDNN behavior and random streams are reset equally for all three runs.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    for position, pos_weight in enumerate(pos_weights):
        run_name = f"pos_weight_{int(pos_weight)}"
        run_dir = out_root / run_name
        run_dir.mkdir(exist_ok=True)
        config = dict(config_common, pos_weight=float(pos_weight))
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        train_generator = torch.Generator().manual_seed(seed)
        val_generator = torch.Generator().manual_seed(seed + 1)
        loaders = {
            "train": DataLoader(CachedSweepDataset(cached["train"], augment=True),
                batch_size=8 if device.type == "cuda" else 4, shuffle=True,
                num_workers=0, generator=train_generator),
            "val": DataLoader(CachedSweepDataset(cached["val"], augment=False),
                batch_size=8 if device.type == "cuda" else 4, shuffle=False,
                num_workers=0, generator=val_generator),
        }
        model = unet_class(base=base).to(device)
        initial_digest = hashlib.sha256()
        for value in model.state_dict().values():
            initial_digest.update(value.detach().cpu().numpy().tobytes())
        config["initial_weights_sha256"] = initial_digest.hexdigest()
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        weight_tensor = torch.tensor([float(pos_weight)], device=device)
        history, top_three = [], []
        best_rank, stale, start_epoch, completed = None, 0, 0, False
        best_patience_ap = -float("inf")
        elapsed_before = 0.0
        latest = run_dir / "resume_latest.pt"
        if resume and latest.exists():
            checkpoint = torch.load(latest, map_location=device, weights_only=False)
            if checkpoint["config"] != config:
                raise ValueError(f"Resume configuration/data mismatch in {run_name}; use a new output directory.")
            model.load_state_dict(checkpoint["state_dict"])
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            _sweep_restore_rng(checkpoint["rng_state"], train_generator, val_generator)
            history = checkpoint["history"]
            top_three = checkpoint["top_three"]
            best_rank = tuple(checkpoint["best_rank"]) if checkpoint["best_rank"] is not None else None
            stale, start_epoch = checkpoint["stale_epochs"], checkpoint["epoch"]
            best_patience_ap = checkpoint["best_patience_ap"]
            elapsed_before, completed = checkpoint["elapsed_seconds"], checkpoint["completed"]
            print(f"Resumed {run_name} at epoch {start_epoch}; completed={completed}", flush=True)
        _sweep_json(run_dir / "config.json", config)
        started = time.perf_counter()
        print(f"Run {position + 1}/3: {run_name}, same initialization={config['initial_weights_sha256'][:12]}, max_epochs={max_epochs}", flush=True)
        for epoch in range(start_epoch + 1, max_epochs + 1):
            if completed:
                break
            epoch_started = time.perf_counter()
            stats = {"epoch": epoch, "pos_weight": float(pos_weight)}
            model.train()
            train_sum, train_count = 0.0, 0
            for x, target, valid in loaders["train"]:
                x, target, valid = [item.to(device) for item in (x, target, valid)]
                optimizer.zero_grad(set_to_none=True)
                logits = model(x)
                loss = loss_function(logits, target, valid, weight_tensor)
                if not bool(torch.isfinite(loss)):
                    raise RuntimeError(f"Non-finite loss in {run_name}, epoch {epoch}; latest checkpoint preserved.")
                loss.backward()
                optimizer.step()
                train_sum += float(loss.item()) * len(x)
                train_count += len(x)
            stats["train_loss"] = train_sum / max(train_count, 1)
            model.eval()
            val_sum, val_count, truth_parts, probability_parts = 0.0, 0, [], []
            with torch.no_grad():
                for x, target, valid in loaders["val"]:
                    x, target, valid = [item.to(device) for item in (x, target, valid)]
                    logits = model(x)
                    loss = loss_function(logits, target, valid, weight_tensor)
                    val_sum += float(loss.item()) * len(x)
                    val_count += len(x)
                    selected = valid >= 0.5
                    truth_parts.append((target[selected] >= 0.5).cpu().numpy())
                    probability_parts.append(torch.sigmoid(logits)[selected].cpu().numpy())
            truth = np.concatenate(truth_parts)
            probability = np.concatenate(probability_parts)
            if not np.isfinite(probability).all():
                raise RuntimeError(f"Non-finite validation probabilities in {run_name}, epoch {epoch}.")
            stats["val_loss"] = val_sum / max(val_count, 1)
            stats["val_ap"] = float(average_precision_score(truth, probability))
            threshold_metrics, threshold_curve = sweep_threshold_metrics(truth, probability)
            stats.update(threshold_metrics)
            stats["epoch_seconds"] = time.perf_counter() - epoch_started
            stats["elapsed_seconds"] = elapsed_before + time.perf_counter() - started
            history.append(dict(stats, threshold_curve=threshold_curve))
            rank = _sweep_rank(stats)
            improved = best_rank is None or rank > best_rank
            if improved:
                best_rank = rank
            if stats["val_ap"] > best_patience_ap + patience_ap_min_delta:
                best_patience_ap, stale = stats["val_ap"], 0
            else:
                stale += 1
            checkpoint_name = f"epoch_{epoch:04d}.pt"
            candidate = dict(stats, checkpoint=str(run_dir / checkpoint_name))
            old_top = top_three
            top_three = sorted(old_top + [candidate], key=_sweep_rank, reverse=True)[:3]
            is_top = any(item["epoch"] == epoch for item in top_three)
            completed = stale >= patience or epoch >= max_epochs
            snapshot = {"state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
                "rng_state": _sweep_rng_state(train_generator, val_generator),
                "config": config, "epoch": epoch, "metrics": stats,
                "history": history, "top_three": top_three,
                "best_rank": best_rank, "best_patience_ap": best_patience_ap, "stale_epochs": stale,
                "elapsed_seconds": stats["elapsed_seconds"], "completed": completed}
            if is_top:
                _sweep_torch_save(run_dir / checkpoint_name, snapshot)
            # Each epoch is resumable. Only the top three validation snapshots persist separately.
            _sweep_torch_save(latest, snapshot)
            keep = {item["checkpoint"] for item in top_three}
            for previous in old_top:
                path = Path(previous["checkpoint"])
                if str(path) not in keep and path.exists():
                    path.unlink()
            _sweep_json(run_dir / "history.json", history)
            _sweep_json(run_dir / "top_three.json", top_three)
            csv_path = run_dir / "history.csv"
            temporary = csv_path.with_suffix(".csv.tmp")
            with temporary.open("w", newline="") as handle:
                writer = csv.DictWriter(handle, fieldnames=list(stats))
                writer.writeheader()
                writer.writerows({key: record[key] for key in stats} for record in history)
            temporary.replace(csv_path)
            if improved or epoch % 5 == 0 or completed:
                print(f"{run_name} epoch {epoch:3d}: AP={stats['val_ap']:.4f}, F1={stats['val_f1']:.4f}, "
                      f"threshold={stats['val_threshold']:.2f}, train_loss={stats['train_loss']:.4f}, "
                      f"val_loss={stats['val_loss']:.4f}, patience={stale}/{patience}", flush=True)
            # The validation probability arrays are temporary and are not saved each epoch.
            del truth, probability, truth_parts, probability_parts
            if completed:
                reason = "patience" if stale >= patience else "epoch budget"
                print(f"Finished {run_name} by {reason}; best AP={best_rank[0]:.4f}.", flush=True)
                break
        result = {"run": run_name, "config": config, "epochs_completed": len(history),
            "top_three": top_three, "history_file": str(run_dir / "history.json"),
            "csv_file": str(run_dir / "history.csv"), "resume_checkpoint": str(latest),
            "elapsed_seconds": elapsed_before + time.perf_counter() - started,
            "completed": completed}
        all_results.append(result)
        _sweep_json(out_root / "run_results.json", all_results)
        del model, optimizer, loaders
        if device.type == "cuda":
            torch.cuda.empty_cache()
    global_ranking = sorted([dict(item, run=run["run"]) for run in all_results
                             for item in run["top_three"]], key=_sweep_rank, reverse=True)
    _sweep_json(out_root / "validation_ranking.json", global_ranking)
    print("Validation sweep finished; selected candidates are ranked in validation_ranking.json. No test files were read.", flush=True)
    return {"runs": all_results, "validation_ranking": global_ranking,
            "output_directory": str(out_root), "data_fingerprint": fingerprint}


In [ ]:
# Run after the existing benchmark cells and validation-sweep helper definitions.
# Cloud training only; preserve the original pilot models and test results.
UNET_LONG_OUT = ROOT / 'unet_validation_sweep'
assert DEVICE.type == 'cuda', 'Connect a Colab GPU before running this experiment.'
print('Colab GPU:', torch.cuda.get_device_name(0), flush=True)
print('Training three weights, max 150 epochs each, patience 20; top 3 per run; validation only.', flush=True)
sweep_results = run_validation_sweep(
    by_split, DEVICE, UNET_LONG_OUT,
    max_epochs=150, patience=20, seed=SEED,
)
print('All three U-Net runs completed. Run the following report cell.', flush=True)


In [ ]:
# Validation-only report. Run after all three longer U-Net runs finish.
import json
import shutil
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve
from google.colab import files

REPORT_OUT = UNET_LONG_OUT / 'report'
REPORT_OUT.mkdir(exist_ok=True)
best_rows = []
for run in sweep_results['runs']:
    best = dict(run['top_three'][0])
    best.update(run=run['run'], epochs_completed=run['epochs_completed'],
                training_seconds=run['elapsed_seconds'])
    best_rows.append(best)
unet_long_table = pd.DataFrame(best_rows)
columns = ['run', 'epochs_completed', 'epoch', 'val_ap', 'val_f1', 'val_iou',
           'val_precision', 'val_recall', 'val_threshold', 'training_seconds']
unet_long_table[columns].to_csv(REPORT_OUT / 'best_per_weight.csv', index=False)
print('VALIDATION RESULTS — best checkpoint selected by AP; no new test evaluation')
display(unet_long_table[columns].round(4))
top9 = pd.DataFrame(sweep_results['validation_ranking'])
top9.to_csv(REPORT_OUT / 'top_9_checkpoints.csv', index=False)
print('Saved checkpoints: 3 independent epoch snapshots per weight, 9 in total.')
display(top9[['run','epoch','val_ap','val_f1','val_iou','val_threshold']].round(4))

histories = {run['run']: pd.read_csv(run['csv_file']) for run in sweep_results['runs']}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for _, row in unet_long_table.iterrows():
    history = histories[row['run']]
    label = row['run'].replace('pos_weight_', 'positive weight ')
    for ax, metric, title in zip(axes, ['val_ap','val_f1','val_iou'],
                                  ['Validation AP','Validation F1','Validation IoU']):
        line, = ax.plot(history['epoch'], history[metric], label=label, linewidth=1.4)
        ax.scatter(row['epoch'], row[metric], color=line.get_color(), s=45, marker='*', zorder=5)
        ax.set(xlabel='Training epoch', ylabel=title, title=title)
        ax.grid(alpha=.2)
axes[0].legend()
fig.suptitle('U-Net longer training — same spatial validation split; star = AP-selected checkpoint')
fig.tight_layout(); fig.savefig(REPORT_OUT/'unet_validation_curves.png', dpi=170); plt.show()
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for ax, (_, row) in zip(axes, unet_long_table.iterrows()):
    history = histories[row['run']]
    ax.plot(history['epoch'], history['train_loss'], label='Train loss')
    ax.plot(history['epoch'], history['val_loss'], label='Validation loss')
    ax.axvline(row['epoch'], color='gray', linestyle=':', label='AP-selected epoch')
    ax.set(title=row['run'].replace('pos_weight_','Positive weight '), xlabel='Epoch', ylabel='BCE + Dice')
    ax.legend(); ax.grid(alpha=.2)
fig.suptitle('Loss diagnostics — compare train/validation within a run; scales differ between weights')
fig.tight_layout(); fig.savefig(REPORT_OUT/'unet_loss_curves.png', dpi=170); plt.show()

# Read only validation tiles and models. Preserve original pilot model objects.
validation_records = by_split['val']
validation_arrays = [read_tile(record) for record in validation_records]
validation_truth_long = np.concatenate([target[valid] for _, target, valid in validation_arrays])
validation_probability_maps = {}
validation_probability_flat = {}
for _, row in unet_long_table.iterrows():
    checkpoint = torch.load(row['checkpoint'], map_location=DEVICE, weights_only=False)
    selected_model = PostEventUNet(base=8).to(DEVICE)
    selected_model.load_state_dict(checkpoint['state_dict']); selected_model.eval()
    maps = []
    with torch.no_grad():
        for start in range(0, len(validation_arrays), 8):
            batch = np.stack([item[0] for item in validation_arrays[start:start+8]])
            probability = torch.sigmoid(selected_model(torch.from_numpy(batch).to(DEVICE)))[:,0].cpu().numpy()
            maps.extend(probability)
    validation_probability_maps[row['run']] = maps
    flat = np.concatenate([probability[valid] for probability, (_, _, valid) in zip(maps, validation_arrays)])
    validation_probability_flat[row['run']] = flat
    assert np.isclose(average_precision_score(validation_truth_long, flat), row['val_ap'], atol=1e-6)
    np.savez_compressed(REPORT_OUT/(row['run']+'_validation_probabilities.npz'),
                        truth=validation_truth_long, probability=flat)
    del selected_model, checkpoint
    torch.cuda.empty_cache()

# Score the existing pilot and RF validation predictions on the same 99-point threshold grid.
comparison_rows = []
for name in ['Random Forest','U-Net']:
    flat = pooled(val_predictions, name)
    assert np.array_equal(pooled(val_predictions, 'truth'), validation_truth_long)
    m, _ = sweep_threshold_metrics(validation_truth_long, flat)
    m.update(model='RF 150 trees' if name=='Random Forest' else 'U-Net original 10 epochs',
             val_ap=float(average_precision_score(validation_truth_long, flat)))
    comparison_rows.append(m)
for _, row in unet_long_table.iterrows():
    comparison_rows.append(dict(row, model=f"U-Net weight {int(row['pos_weight'])}, epoch {int(row['epoch'])}"))
validation_comparison_table = pd.DataFrame(comparison_rows)
compare_cols = ['model','val_ap','val_f1','val_iou','val_precision','val_recall','val_threshold']
validation_comparison_table[compare_cols].to_csv(REPORT_OUT/'validation_model_comparison.csv',index=False)
print('Same validation pixels; common threshold grid 0.01–0.99:')
display(validation_comparison_table[compare_cols].round(4))
fig, axes = plt.subplots(1, 2, figsize=(13,4.5))
for _, row in unet_long_table.iterrows():
    flat = validation_probability_flat[row['run']]
    precision, recall, _ = precision_recall_curve(validation_truth_long, flat)
    _, curve = sweep_threshold_metrics(validation_truth_long, flat)
    label = row['run'].replace('pos_weight_','weight ')
    axes[0].plot(recall, precision, label=f"{label}; AP {row['val_ap']:.3f}")
    axes[1].plot([r['threshold'] for r in curve], [r['f1'] for r in curve], label=label)
axes[0].set(xlabel='Recall',ylabel='Precision',title='Validation precision–recall curves')
axes[1].set(xlabel='Probability threshold',ylabel='F1',title='Validation threshold sensitivity')
for ax in axes: ax.legend(); ax.grid(alpha=.2)
fig.tight_layout(); fig.savefig(REPORT_OUT/'unet_pr_threshold_curves.png',dpi=170); plt.show()

# Visual examples deliberately include mapped positives; these are qualitative validation examples.
example_indices = sorted(range(len(validation_records)),
                         key=lambda i: validation_records[i]['positive_pixels'], reverse=True)[:3]
fig, axes = plt.subplots(3, 7, figsize=(22,10))
display_names = ['Post-event RGB','GNS target','RF 150','U-Net 10'] + [r['run'].replace('pos_weight_','U-Net w') for _,r in unet_long_table.iterrows()]
pilot_thresholds = {r['model']: r['val_threshold'] for r in comparison_rows[:2]}
for visual_row, i in enumerate(example_indices):
    x, target, valid = validation_arrays[i]
    rgb = np.moveaxis(x[[2,1,0]],0,-1)
    rgb = np.clip(rgb/.3,0,1)
    images = [rgb, np.where(valid, target.astype(float), np.nan)]
    for name, label in [('Random Forest','RF 150 trees'), ('U-Net','U-Net original 10 epochs')]:
        images.append(np.where(valid, val_predictions[i][name] >= pilot_thresholds[label], np.nan))
    for _, row in unet_long_table.iterrows():
        images.append(np.where(valid, validation_probability_maps[row['run']][i] >= row['val_threshold'], np.nan))
    for j, image in enumerate(images):
        if j==0: axes[visual_row,j].imshow(image)
        else: axes[visual_row,j].imshow(image,cmap='gray',vmin=0,vmax=1)
        axes[visual_row,j].set_title(display_names[j] if visual_row==0 else '')
        axes[visual_row,j].set_xticks([]); axes[visual_row,j].set_yticks([])
    axes[visual_row,0].set_ylabel(validation_records[i]['tile'],fontsize=9)
fig.suptitle('Validation tiles with most mapped positives — white = landslide; same examples for every model')
fig.tight_layout(); fig.savefig(REPORT_OUT/'validation_prediction_comparison.png',dpi=170); plt.show()

winner = sweep_results['validation_ranking'][0]
feedback = [
    f"Best validation AP: {winner['run']}, epoch {winner['epoch']}; AP={winner['val_ap']:.4f}, F1={winner['val_f1']:.4f}, IoU={winner['val_iou']:.4f}, threshold={winner['val_threshold']:.2f}.",
    'Nine independently saved epoch checkpoints: top three from each positive-weight run.',
    'These checkpoints are snapshots from three single-seed runs, not nine independent experiments.',
    'Validation data were used to choose epochs, weights and thresholds; results are validation performance, not unbiased test performance.',
    'No test files were read during this sweep or report. The earlier pilot test remains unchanged.',
    'GNS labels represent mapped event landslides, not confirmed blocked roads. Inventory background may contain omissions.',
]
for run in sweep_results['runs']:
    best = run['top_three'][0]
    feedback.append(f"{run['run']}: stopped after {run['epochs_completed']} epochs; best at {best['epoch']}; elapsed {run['elapsed_seconds']:.1f}s.")
    if run['epochs_completed']==150:
        feedback.append(f"{run['run']}: reached epoch budget; the curve must be checked before claiming convergence.")
    elif best['epoch'] < run['epochs_completed']-15:
        feedback.append(f"{run['run']}: further training did not surpass the earlier validation AP; use the saved best checkpoint.")
feedback_text = '\n'.join(feedback)
print(feedback_text)
(REPORT_OUT/'feedback.txt').write_text(feedback_text)
(REPORT_OUT / "checkpoint_README.txt").write_text("Kaikōura post-event U-Net validation sweep\n\nRun the benchmark helper-definition cell before loading a checkpoint.\nThe model is PostEventUNet(in_channels=8, base=8). Inputs, in order:\nB2, B3, B4, B8, B11, B12, elevation, slope.\nUse the benchmark's normalize_features(): optical reflectance clipped to [0, 1];\nelevation clipped to [0, 3000] then divided by 3000; slope clipped to [0, 90]\nthen divided by 90. Tiles use the same 10 m NZTM (EPSG:2193) grid.\nExclude cloud, ocean, missing-input and outside-AOI pixels using the valid mask.\n\nEach pos_weight_20 / pos_weight_5 / pos_weight_1 directory contains its three\nhighest-ranked validation checkpoints (epoch_XXXX.pt), top_three.json,\nhistory.csv/history.json, and resume_latest.pt. Ranking uses validation average\nprecision, then validation F1, then the earlier epoch. Binary thresholds are\nchosen on validation only from the common 0.01–0.99 grid. Read the saved threshold\nfrom each checkpoint; do not assume 0.5. validation_ranking.json ranks the candidates.\n\nFor your own trusted checkpoint files only:\n\n    checkpoint = torch.load(path, map_location=device, weights_only=False)\n    model = PostEventUNet(in_channels=8, base=8).to(device)\n    model.load_state_dict(checkpoint['state_dict'])\n    model.eval()\n    threshold = checkpoint['metrics']['val_threshold']\n    with torch.no_grad():\n        probability = torch.sigmoid(model(normalized_input_tensor))\n    prediction = probability >= threshold\n\nnormalized_input_tensor must have shape [batch, 8, height, width]. Preserve the\nvalidity mask, CRS and transform when exporting predictions. Checkpoints also\ncontain optimizer and random-generator state; resume_latest.pt is for continuing\nan interrupted run under the same configuration and frozen train/validation data.\n\nTarget: the union of GNS V3 mapped landslide source and debris areas for the\n2016 Kaikōura earthquake. Background means outside this inventory, not verified\nstable ground. Predictions are not confirmed road blockage or safe passage.\nThe three snapshots within each weight setting come from one training trajectory;\nthey are not independent seeds or three independent replications.\n")
(REPORT_OUT / "landslide_benchmark.py").write_text('"""Shared post-event inputs, compact U-Net, and RF benchmark helpers.\n\nThe Colab notebook embeds this module so it can run independently.\nTargets are mapped GNS inventory, not confirmed road closure or stable ground.\n"""\nfrom pathlib import Path\nimport json\nimport random\nimport time\n\nimport numpy as np\nimport rasterio\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\nFEATURE_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "elevation", "slope"]\n\n\ndef normalize_features(arr):\n    """Same fixed, label-independent transform for both models."""\n    x = np.nan_to_num(arr[:8].astype("float32"), nan=0, posinf=0, neginf=0)\n    x[:6] = np.clip(x[:6], 0, 1)\n    x[6] = np.clip(x[6], 0, 3000) / 3000.0\n    x[7] = np.clip(x[7], 0, 90) / 90.0\n    return x\n\n\ndef read_tile(record):\n    with rasterio.open(record["path"]) as ds:\n        arr = ds.read()\n    return normalize_features(arr), (arr[8] > 0.5), (arr[9] > 0.5)\n\n\nclass TileDataset(Dataset):\n    def __init__(self, records, augment=False):\n        self.records, self.augment = records, augment\n\n    def __len__(self):\n        return len(self.records)\n\n    def __getitem__(self, i):\n        x, y, valid = read_tile(self.records[i])\n        y, valid = y[None].astype("float32"), valid[None].astype("float32")\n        if self.augment:\n            if random.random() < 0.5:\n                x, y, valid = x[:, :, ::-1], y[:, :, ::-1], valid[:, :, ::-1]\n            if random.random() < 0.5:\n                x, y, valid = x[:, ::-1, :], y[:, ::-1, :], valid[:, ::-1, :]\n            k = random.randrange(4)\n            x, y, valid = [np.rot90(a, k, axes=(1, 2)) for a in (x, y, valid)]\n        return tuple(torch.from_numpy(a.copy()) for a in (x, y, valid))\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_channels, out_channels):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 3, padding=1),\n            nn.GroupNorm(4, out_channels), nn.ReLU(inplace=True),\n            nn.Conv2d(out_channels, out_channels, 3, padding=1),\n            nn.GroupNorm(4, out_channels), nn.ReLU(inplace=True),\n        )\n\n    def forward(self, x):\n        return self.net(x)\n\n\nclass PostEventUNet(nn.Module):\n    def __init__(self, in_channels=8, base=8):\n        super().__init__()\n        self.enc = nn.ModuleList([\n            ConvBlock(in_channels, base), ConvBlock(base, base * 2),\n            ConvBlock(base * 2, base * 4), ConvBlock(base * 4, base * 8),\n        ])\n        self.pool = nn.MaxPool2d(2)\n        self.dec = nn.ModuleList([\n            ConvBlock(base * 8 + base * 4, base * 4),\n            ConvBlock(base * 4 + base * 2, base * 2),\n            ConvBlock(base * 2 + base, base),\n        ])\n        self.head = nn.Conv2d(base, 1, 1)\n\n    def forward(self, x):\n        skips = []\n        for i, block in enumerate(self.enc):\n            x = block(self.pool(x) if i else x)\n            skips.append(x)\n        for block, skip in zip(self.dec, reversed(skips[:-1])):\n            x = nn.functional.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)\n            x = block(torch.cat([x, skip], dim=1))\n        return self.head(x)\n\n\ndef masked_loss(logits, target, valid, pos_weight):\n    bce = nn.functional.binary_cross_entropy_with_logits(\n        logits, target, reduction="none", pos_weight=pos_weight)\n    bce = (bce * valid).sum() / valid.sum().clamp_min(1)\n    p, t = torch.sigmoid(logits) * valid, target * valid\n    dice_loss = 1 - (2 * (p * t).sum() + 1) / (p.sum() + t.sum() + 1)\n    return bce + dice_loss\n\n\ndef confusion_counts(truth, probability, threshold):\n    truth, pred = np.asarray(truth).astype(bool), np.asarray(probability) >= threshold\n    return np.array([np.sum(~truth & ~pred), np.sum(~truth & pred),\n                     np.sum(truth & ~pred), np.sum(truth & pred)], dtype="int64")\n\n\ndef metrics_from_counts(counts):\n    tn, fp, fn, tp = [int(v) for v in counts]\n    precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)\n    specificity = tn / max(tn + fp, 1)\n    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp,\n            "precision": precision, "recall": recall,\n            "f1": 2 * tp / max(2 * tp + fp + fn, 1),\n            "iou": tp / max(tp + fp + fn, 1),\n            "balanced_accuracy": (recall + specificity) / 2,\n            "accuracy": (tp + tn) / max(sum(counts), 1),\n            "positive_fraction": (tp + fn) / max(sum(counts), 1)}\n\n\ndef pixel_metrics(truth, probability, threshold):\n    result = metrics_from_counts(confusion_counts(truth, probability, threshold))\n    result["threshold"] = float(threshold)\n    if len(np.unique(truth)) == 2:\n        result["average_precision"] = float(average_precision_score(truth, probability))\n        result["roc_auc"] = float(roc_auc_score(truth, probability))\n    return result\n\n\ndef select_threshold(truth, probability):\n    """Call on validation only. Deterministic F1 selection; test stays untouched."""\n    scores = [(float(t), metrics_from_counts(confusion_counts(truth, probability, t))["f1"])\n              for t in np.linspace(0.05, 0.95, 19)]\n    chosen = max(scores, key=lambda pair: (pair[1], -abs(pair[0] - 0.5)))\n    return chosen[0], scores\n\n\ndef train_unet(records_by_split, device, out, epochs=10, seed=761, base=8):\n    torch.manual_seed(seed)\n    random.seed(seed)\n    loaders = {split: DataLoader(TileDataset(items, augment=split == "train"),\n                                batch_size=8 if device.type == "cuda" else 4,\n                                shuffle=split == "train", num_workers=0)\n               for split, items in records_by_split.items() if split != "test"}\n    model = PostEventUNet(base=base).to(device)\n    npos = sum(r["positive_pixels"] for r in records_by_split["train"])\n    nvalid = sum(r["valid_pixels"] for r in records_by_split["train"])\n    weight = torch.tensor([min((nvalid - npos) / max(npos, 1), 20)], device=device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)\n    best, stale, history = float("inf"), 0, []\n    start = time.perf_counter()\n    print(f"U-Net: {sum(p.numel() for p in model.parameters()):,} parameters; device={device}; pos_weight={float(weight.item()):.2f}", flush=True)\n    for epoch in range(epochs):\n        stats = {"epoch": epoch + 1}\n        for split in ("train", "val"):\n            model.train(split == "train")\n            loss_sum, count = 0, 0\n            for step, batch in enumerate(loaders[split]):\n                x, y, valid = [a.to(device) for a in batch]\n                with torch.set_grad_enabled(split == "train"):\n                    logits = model(x)\n                    loss = masked_loss(logits, y, valid, weight)\n                    if split == "train":\n                        optimizer.zero_grad(set_to_none=True)\n                        loss.backward()\n                        optimizer.step()\n                loss_sum += loss.item() * len(x)\n                count += len(x)\n                if split == "train" and (step + 1) % 5 == 0:\n                    print(f"  epoch {epoch + 1}: batch {step + 1}/{len(loaders[split])}", flush=True)\n            stats[split + "_loss"] = loss_sum / max(count, 1)\n        history.append(stats)\n        print(f"Epoch {epoch + 1}/{epochs}: train={stats[\'train_loss\']:.4f}, val={stats[\'val_loss\']:.4f}", flush=True)\n        if stats["val_loss"] < best - 1e-5:\n            best, stale = stats["val_loss"], 0\n            torch.save({"state_dict": model.state_dict(), "epoch": epoch + 1,\n                        "base": base, "val_loss": best, "features": FEATURE_NAMES}, out / "unet_best.pt")\n        else:\n            stale += 1\n        if stale >= 4:\n            print("Early stopping: validation loss did not improve for four epochs.", flush=True)\n            break\n    checkpoint = torch.load(out / "unet_best.pt", map_location=device, weights_only=False)\n    model.load_state_dict(checkpoint["state_dict"])\n    model.eval()\n    elapsed = time.perf_counter() - start\n    (out / "training_history.json").write_text(json.dumps(history, indent=2))\n    print(f"Best epoch {checkpoint[\'epoch\']}; training {elapsed:.1f} seconds.", flush=True)\n    return model, history, elapsed\n\n\ndef sample_rf_pixels(records, max_pixels=150_000, seed=761):\n    rng = np.random.default_rng(seed)\n    features, labels = [], []\n    per_tile = max(1, int(np.ceil(max_pixels / len(records))))\n    for record in records:\n        x, y, valid = read_tile(record)\n        indices = np.flatnonzero(valid.ravel())\n        if len(indices) > per_tile:\n            indices = rng.choice(indices, per_tile, replace=False)\n        features.append(x.reshape(8, -1).T[indices])\n        labels.append(y.ravel()[indices].astype("uint8"))\n    x, y = np.concatenate(features), np.concatenate(labels)\n    if len(y) > max_pixels:\n        keep = rng.choice(len(y), max_pixels, replace=False)\n        x, y = x[keep], y[keep]\n    if len(np.unique(y)) < 2:\n        raise ValueError("The sampled RF training pixels have only one class; increase RF_MAX_PIXELS.")\n    return x, y\n\n\ndef train_rf(records, out, max_pixels=150_000, trees=150, seed=761):\n    import joblib\n    x, y = sample_rf_pixels(records, max_pixels, seed)\n    model = RandomForestClassifier(n_estimators=trees, max_depth=20,\n        min_samples_leaf=5, max_features="sqrt", class_weight="balanced_subsample",\n        n_jobs=-1, random_state=seed)\n    print(f"RF: {len(y):,} training pixels; mapped-positive fraction={y.mean():.4f}; trees={trees}", flush=True)\n    start = time.perf_counter()\n    model.fit(x, y)\n    elapsed = time.perf_counter() - start\n    joblib.dump(model, out / "rf_model.joblib")\n    print(f"RF training {elapsed:.1f} seconds.", flush=True)\n    return model, elapsed, {"sampled_pixels": int(len(y)), "positive_pixels": int(y.sum())}\n\n\ndef tile_probabilities(record, unet, rf, device):\n    x, y, valid = read_tile(record)\n    with torch.no_grad():\n        p_unet = torch.sigmoid(unet(torch.from_numpy(x)[None].to(device)))[0, 0].cpu().numpy()\n    p_rf = np.zeros(y.shape, dtype="float32")\n    if valid.any():\n        features = x.reshape(8, -1).T[valid.ravel()]\n        p_rf[valid] = rf.predict_proba(features)[:, list(rf.classes_).index(1)]\n    return {"truth": y, "valid": valid, "U-Net": p_unet, "Random Forest": p_rf}\n\n\ndef predict_split(records, unet, rf, device):\n    results = []\n    for i, record in enumerate(records):\n        results.append(tile_probabilities(record, unet, rf, device))\n        if (i + 1) % 5 == 0 or i + 1 == len(records):\n            print(f"Predicted {i + 1}/{len(records)} {record[\'split\']} tiles", flush=True)\n    return results\n\n\ndef pooled(results, key):\n    return np.concatenate([item[key][item["valid"]] for item in results])\n\n\ndef bootstrap_f1(results, key, threshold, seed=761, repetitions=1000):\n    """Tile-resampling interval; spatial dependence still limits interpretation."""\n    counts = np.stack([confusion_counts(item["truth"][item["valid"]],\n                       item[key][item["valid"]], threshold) for item in results])\n    rng = np.random.default_rng(seed)\n    values = [metrics_from_counts(counts[rng.integers(0, len(counts), len(counts))].sum(0))["f1"]\n              for _ in range(repetitions)]\n    return [float(v) for v in np.percentile(values, [2.5, 97.5])]\n')
bundle_path = ROOT/'kaikoura_unet_long_validation_outputs.zip'
with zipfile.ZipFile(bundle_path,'w',compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in UNET_LONG_OUT.rglob('*'):
        if path.is_file() and path.suffix!='.tmp':
            bundle.write(path,path.relative_to(UNET_LONG_OUT))
print('Report and checkpoint bundle:',bundle_path, 'bytes:',bundle_path.stat().st_size)
files.download(str(bundle_path))
